In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load cleaned Silver data
ml_source_df = spark.table(
    "workspace.silver.scada_lakeflow_silver"
)

# Order readings separately for each sensor
sensor_window = (
    Window
    .partitionBy("id")
    .orderBy("timestamp")
)

# Previous 10 readings, excluding the current reading
history_window = (
    Window
    .partitionBy("id")
    .orderBy("timestamp")
    .rowsBetween(-10, -1)
)

ml_features_df = (
    ml_source_df

    # Previous sensor reading
    .withColumn(
        "previous_value",
        F.lag("value", 1).over(sensor_window)
    )

    # Change from previous reading
    .withColumn(
        "value_change",
        F.col("value") - F.col("previous_value")
    )

    # Rolling statistics
    .withColumn(
        "rolling_avg_10",
        F.avg("value").over(history_window)
    )

    .withColumn(
        "rolling_std_10",
        F.stddev("value").over(history_window)
    )

    .withColumn(
        "rolling_min_10",
        F.min("value").over(history_window)
    )

    .withColumn(
        "rolling_max_10",
        F.max("value").over(history_window)
    )

    # Time-based features
    .withColumn(
        "hour",
        F.hour("timestamp")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("timestamp")
    )
)

display(ml_features_df.limit(20))

id,value,unit,timestamp,previous_value,value_change,rolling_avg_10,rolling_std_10,rolling_min_10,rolling_max_10,hour,day_of_week
19,31.978,mg,2018-04-23T04:53:23.137Z,null,null,null,null,null,null,4,2
19,81.114,mg,2018-04-23T04:53:25.947Z,31.978,49.136,31.978,null,31.978,31.978,4,2
19,6.202,mg,2018-04-23T04:53:28.760Z,81.114,-74.912,56.54600000000001,34.7443988003822,31.978,81.114,4,2
19,17.668,mg,2018-04-23T04:53:31.573Z,6.202,11.466,39.76466666666667,38.05819319585907,6.202,81.114,4,2
19,25.39,mg,2018-04-23T04:53:34.387Z,17.668,7.722000000000001,34.240500000000004,32.98004016067901,6.202,81.114,4,2
19,114.647,mg,2018-04-23T04:53:37.183Z,25.39,89.257,32.470400000000005,28.83450288803329,6.202,81.114,4,2
19,18.906,mg,2018-04-23T04:53:39.997Z,114.647,-95.74100000000001,46.166500000000006,42.315975431271816,6.202,114.647,4,2
19,9.052,mg,2018-04-23T04:53:42.810Z,18.906,-9.854,42.27214285714286,39.97953981071056,6.202,114.647,4,2
19,38.753,mg,2018-04-23T04:53:45.623Z,9.052,29.701,38.119625000000006,38.832630380491324,6.202,114.647,4,2
19,9.463,mg,2018-04-23T04:53:48.433Z,38.753,-29.29,38.190000000000005,36.32521312050351,6.202,114.647,4,2


In [0]:
from pyspark.sql import functions as F

ml_features_df = (
    ml_features_df
    .withColumn(
        "z_score",
        F.when(
            F.col("rolling_std_10").isNull() |
            (F.col("rolling_std_10") == 0),
            None
        ).otherwise(
            (
                F.col("value") - F.col("rolling_avg_10")
            ) / F.col("rolling_std_10")
        )
    )
)

display(
    ml_features_df.select(
        "id",
        "timestamp",
        "value",
        "rolling_avg_10",
        "rolling_std_10",
        "z_score"
    ).limit(30)
)

id,timestamp,value,rolling_avg_10,rolling_std_10,z_score
19,2018-04-23T04:53:23.137Z,31.978,null,null,null
19,2018-04-23T04:53:25.947Z,81.114,31.978,null,null
19,2018-04-23T04:53:28.760Z,6.202,56.54600000000001,34.7443988003822,-1.448981756433391
19,2018-04-23T04:53:31.573Z,17.668,39.76466666666667,38.05819319585907,-0.5806020940865607
19,2018-04-23T04:53:34.387Z,25.39,34.240500000000004,32.98004016067901,-0.26835928509729823
19,2018-04-23T04:53:37.183Z,114.647,32.470400000000005,28.83450288803329,2.8499398903840447
19,2018-04-23T04:53:39.997Z,18.906,46.166500000000006,42.315975431271816,-0.6442129650130739
19,2018-04-23T04:53:42.810Z,9.052,42.27214285714286,39.97953981071056,-0.8309285953372366
19,2018-04-23T04:53:45.623Z,38.753,38.119625000000006,38.832630380491324,0.01631038108400166
19,2018-04-23T04:53:48.433Z,9.463,38.190000000000005,36.32521312050351,-0.7908281199810842


In [0]:
from pyspark.sql import functions as F

ml_features_df = (
    ml_features_df
    .withColumn(
        "anomaly_score",
        F.abs(F.col("z_score"))
    )
    .withColumn(
        "is_anomaly",
        F.when(
            F.abs(F.col("z_score")) >= 3,
            1
        ).otherwise(0)
    )
)

display(
    ml_features_df.select(
        "id",
        "timestamp",
        "value",
        "rolling_avg_10",
        "rolling_std_10",
        "z_score",
        "anomaly_score",
        "is_anomaly"
    ).orderBy(F.desc("anomaly_score")).limit(30)
)

id,timestamp,value,rolling_avg_10,rolling_std_10,z_score,anomaly_score,is_anomaly
32,2018-04-23T05:32:19.663Z,19451.834,30.374100000000006,1.7126731185813342,11339.852123146218,11339.852123146218,1
32,2018-04-23T09:44:38.500Z,20064.223,31.108800000000002,2.3237353454202907,8621.082533982217,8621.082533982217,1
62,2018-04-23T05:32:19.663Z,16095.72,50.949600000000004,1.9320508504925262,8304.528007588311,8304.528007588311,1
44,2018-04-23T09:44:38.500Z,15552.09,50.85410000000001,2.7758095695970675,5584.40177949604,5584.40177949604,1
62,2018-04-23T09:44:38.500Z,27863.16,65.8915,5.0922830123411025,5458.7045599456205,5458.7045599456205,1
44,2018-04-23T05:32:19.663Z,16400.475,59.65560000000001,4.916682784154375,3323.545593110797,3323.545593110797,1
62,2018-04-23T05:56:08.707Z,18016.135,66.13640000000001,5.415847285718292,3314.347257784491,3314.347257784491,1
44,2018-04-23T05:56:08.707Z,21224.289,65.2788,6.422231574772121,3294.650769542016,3294.650769542016,1
44,2018-04-23T07:32:00.487Z,19842.912,69.191,7.2937078666782655,2711.0656694021723,2711.0656694021723,1
62,2018-04-23T07:32:00.487Z,25803.125,72.9954,12.532754757922227,2053.030646254003,2053.030646254003,1


In [0]:
from pyspark.sql import functions as F

anomaly_summary = (
    ml_features_df
    .filter(F.col("is_anomaly") == 1)
    .groupBy("id")
    .agg(
        F.count("*").alias("anomaly_count"),
        F.max("anomaly_score").alias("max_anomaly_score"),
        F.avg("anomaly_score").alias("avg_anomaly_score")
    )
    .orderBy(F.desc("anomaly_count"))
)

print(
    "Total anomalies:",
    ml_features_df.filter(F.col("is_anomaly") == 1).count()
)

display(anomaly_summary)

Total anomalies: 18741


id,anomaly_count,max_anomaly_score,avg_anomaly_score
32,1937,11339.852123146218,20.750278956211258
68,1832,991.036758365716,7.814363834278768
109,1809,412.99540982879154,6.439165673223982
64,1796,250.0280876429943,8.251807297940815
92,1772,759.2727528677744,8.229786879771174
44,1725,5584.40177949604,16.590680434449993
47,1671,286.2969039610031,7.163868726026561
46,1669,105.35396038511396,7.483816901085982
62,1629,8304.528007588311,21.304524039859015
105,561,375.35755648406706,7.455413561431757


In [0]:
(
    ml_features_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.scada_ml_features")
)

print("ML feature table saved")

print(
    "Feature rows:",
    spark.table("workspace.silver.scada_ml_features").count()
)

ML feature table saved
Feature rows: 282866


In [0]:
from pyspark.sql import functions as F

feature_columns = [
    "value",
    "value_change",
    "rolling_avg_10",
    "rolling_std_10",
    "rolling_min_10",
    "rolling_max_10",
    "hour",
    "day_of_week"
]

model_ready_df = (
    spark.table("workspace.silver.scada_ml_features")
    .dropna(subset=feature_columns)
    .select(
        "id",
        "timestamp",
        "unit",
        *feature_columns,
        "z_score",
        "is_anomaly"
    )
)

print("Model-ready rows:", model_ready_df.count())

model_ready_df.printSchema()

Model-ready rows: 282802
root
 |-- id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)
 |-- value_change: double (nullable = true)
 |-- rolling_avg_10: double (nullable = true)
 |-- rolling_std_10: double (nullable = true)
 |-- rolling_min_10: double (nullable = true)
 |-- rolling_max_10: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- z_score: double (nullable = true)
 |-- is_anomaly: integer (nullable = true)



In [0]:
feature_columns = [
    "value",
    "value_change",
    "rolling_avg_10",
    "rolling_std_10",
    "rolling_min_10",
    "rolling_max_10",
    "hour",
    "day_of_week"
]

training_df = (
    model_ready_df
    .select(*feature_columns)
    .sample(
        withReplacement=False,
        fraction=0.40,
        seed=42
    )
    .limit(100000)
)

training_pdf = training_df.toPandas()

print("Training rows:", len(training_pdf))
print("Features:", training_pdf.shape[1])

training_pdf.head()

Training rows: 100000
Features: 8


,value,value_change,rolling_avg_10,rolling_std_10,rolling_min_10,rolling_max_10,hour,day_of_week
0,6.202,-74.912,56.546000,34.744399,31.978,81.114,4,2
1,17.668,11.466,39.764667,38.058193,6.202,81.114,4,2
2,25.390,7.722,34.240500,32.980040,6.202,81.114,4,2
3,114.647,89.257,32.470400,28.834503,6.202,81.114,4,2
4,18.906,-95.741,46.166500,42.315975,6.202,114.647,4,2


In [0]:
import mlflow
import mlflow.sklearn

from sklearn.ensemble import IsolationForest


# Train Isolation Forest
model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

with mlflow.start_run() as run:

    model.fit(training_pdf)

    # Predictions on training data
    #  1 = normal
    # -1 = anomaly
    predictions = model.predict(training_pdf)

    anomaly_count = (predictions == -1).sum()
    anomaly_rate = anomaly_count / len(predictions)

    # Log model configuration
    mlflow.log_param("model_type", "IsolationForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("contamination", "auto")
    mlflow.log_param("training_rows", len(training_pdf))
    mlflow.log_param("feature_count", len(feature_columns))

    # Log useful training metrics
    mlflow.log_metric(
        "training_anomaly_count",
        int(anomaly_count)
    )

    mlflow.log_metric(
        "training_anomaly_rate",
        float(anomaly_rate)
    )

    # Save model into MLflow
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="isolation_forest_model"
    )

    run_id = run.info.run_id


print("MLflow Run ID:", run_id)
print("Training anomalies:", anomaly_count)
print("Training anomaly rate:", round(anomaly_rate * 100, 2), "%")

2026/09/24 16:27:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-efb753c8-7284.cloud.databricks.com/ml/experiments/4223316458238612/models/m-367372f3462c4404b1444285738ff1ee?o=7474652842974359
2026/09/24 16:27:44 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


MLflow Run ID: 83619b5d0dc9489f8878e1ec7a3e98fe
Training anomalies: 10597
Training anomaly rate: 10.6 %


In [0]:
# Bring the complete model-ready dataset to pandas for sklearn scoring
full_pdf = model_ready_df.select(
    "id",
    "timestamp",
    "unit",
    *feature_columns,
    "z_score",
    "is_anomaly"
).toPandas()

X_full = full_pdf[feature_columns]

# Isolation Forest prediction:
#  1 = normal
# -1 = anomaly
full_pdf["ml_prediction"] = model.predict(X_full)

# Higher value = more anomalous
full_pdf["ml_anomaly_score"] = -model.decision_function(X_full)

full_pdf["ml_is_anomaly"] = (
    full_pdf["ml_prediction"] == -1
).astype(int)

ml_anomaly_count = full_pdf["ml_is_anomaly"].sum()
ml_anomaly_rate = ml_anomaly_count / len(full_pdf)

print("Full dataset rows:", len(full_pdf))
print("ML anomalies:", ml_anomaly_count)
print("ML anomaly rate:", round(ml_anomaly_rate * 100, 2), "%")

Full dataset rows: 282802
ML anomalies: 26962
ML anomaly rate: 9.53 %


In [0]:
import pandas as pd

comparison = pd.crosstab(
    full_pdf["is_anomaly"],
    full_pdf["ml_is_anomaly"],
    rownames=["Z-score anomaly"],
    colnames=["ML anomaly"]
)

print(comparison)

both_anomaly = (
    (full_pdf["is_anomaly"] == 1) &
    (full_pdf["ml_is_anomaly"] == 1)
).sum()

z_only = (
    (full_pdf["is_anomaly"] == 1) &
    (full_pdf["ml_is_anomaly"] == 0)
).sum()

ml_only = (
    (full_pdf["is_anomaly"] == 0) &
    (full_pdf["ml_is_anomaly"] == 1)
).sum()

print("Both methods flagged:", both_anomaly)
print("Z-score only:", z_only)
print("ML only:", ml_only)

ML anomaly            0      1
Z-score anomaly               
0                241098  22963
1                 14742   3999
Both methods flagged: 3999
Z-score only: 14742
ML only: 22963


In [0]:
import numpy as np

full_pdf["anomaly_category"] = np.select(
    [
        (full_pdf["is_anomaly"] == 1) &
        (full_pdf["ml_is_anomaly"] == 1),

        (full_pdf["is_anomaly"] == 1) &
        (full_pdf["ml_is_anomaly"] == 0),

        (full_pdf["is_anomaly"] == 0) &
        (full_pdf["ml_is_anomaly"] == 1)
    ],
    [
        "HIGH_CONFIDENCE",
        "Z_SCORE_ONLY",
        "ML_ONLY"
    ],
    default="NORMAL"
)

print(
    full_pdf["anomaly_category"]
    .value_counts()
)

anomaly_category
NORMAL             241098
ML_ONLY             22963
Z_SCORE_ONLY        14742
HIGH_CONFIDENCE      3999
Name: count, dtype: int64


In [0]:
import numpy as np

full_pdf["anomaly_category"] = np.select(
    [
        (full_pdf["is_anomaly"] == 1) &
        (full_pdf["ml_is_anomaly"] == 1),

        (full_pdf["is_anomaly"] == 1) &
        (full_pdf["ml_is_anomaly"] == 0),

        (full_pdf["is_anomaly"] == 0) &
        (full_pdf["ml_is_anomaly"] == 1)
    ],
    [
        "HIGH_CONFIDENCE",
        "Z_SCORE_ONLY",
        "ML_ONLY"
    ],
    default="NORMAL"
)

print(full_pdf["anomaly_category"].value_counts())

anomaly_category
NORMAL             241098
ML_ONLY             22963
Z_SCORE_ONLY        14742
HIGH_CONFIDENCE      3999
Name: count, dtype: int64


In [0]:
from pyspark.sql import functions as F

# Convert pandas results back to Spark
anomaly_spark_df = spark.createDataFrame(full_pdf)

# Keep useful analytics/model fields with clearer naming
anomaly_gold_df = anomaly_spark_df.select(
    "id",
    "timestamp",
    "unit",
    "value",
    "value_change",
    "rolling_avg_10",
    "rolling_std_10",
    "rolling_min_10",
    "rolling_max_10",
    "hour",
    "day_of_week",
    "z_score",
    F.col("is_anomaly").alias("zscore_is_anomaly"),
    "ml_anomaly_score",
    "ml_is_anomaly",
    "anomaly_category"
)

(
    anomaly_gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.scada_anomaly_results")
)

print("Anomaly Gold table saved")
print("Rows:", spark.table(
    "workspace.gold.scada_anomaly_results"
).count())

Anomaly Gold table saved
Rows: 282802


In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Small example of the model input
input_example = training_pdf[feature_columns].head(5)

# Infer expected model input/output schema
signature = infer_signature(
    input_example,
    model.predict(input_example)
)

with mlflow.start_run() as run:

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="scada_isolation_forest",
        signature=signature,
        input_example=input_example
    )

    signed_run_id = run.info.run_id

print("Signed MLflow Run ID:", signed_run_id)
print("Model URI:", model_info.model_uri)

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-efb753c8-7284.cloud.databricks.com/ml/experiments/4223316458238612/models/m-a1a12c2911a14ea3a956aa7cc43503b2?o=7474652842974359


Signed MLflow Run ID: c472a777f6bd4bff87e02e01d0574893
Model URI: models:/m-a1a12c2911a14ea3a956aa7cc43503b2


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ml
""")

print("ML schema ready")

ML schema ready


In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="workspace.ml.scada_isolation_forest"
)

print("Registered model:", registered_model.name)
print("Model version:", registered_model.version)

Successfully registered model 'workspace.ml.scada_isolation_forest'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered model: workspace.ml.scada_isolation_forest
Model version: 1


🔗 Created version '1' of model 'workspace.ml.scada_isolation_forest': https://dbc-efb753c8-7284.cloud.databricks.com/explore/data/models/workspace/ml/scada_isolation_forest/version/1?o=7474652842974359


In [0]:
import mlflow

model_name = "workspace.ml.scada_isolation_forest"
model_version = registered_model.version

model_uri = f"models:/{model_name}/{model_version}"

# Load model back from Unity Catalog
loaded_model = mlflow.sklearn.load_model(model_uri)

# Test on a few real feature rows
test_sample = training_pdf[feature_columns].head(10)

predictions = loaded_model.predict(test_sample)

print("Model:", model_name)
print("Version:", model_version)
print("Predictions:", predictions)

Model: workspace.ml.scada_isolation_forest
Version: 1
Predictions: [1 1 1 1 1 1 1 1 1 1]


In [0]:
from mlflow import MlflowClient

client = MlflowClient()

model_name = "workspace.ml.scada_isolation_forest"

client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version="1"
)

print("Champion alias assigned to Version 1")

Champion alias assigned to Version 1


In [0]:
champion_model = mlflow.sklearn.load_model(
    "models:/workspace.ml.scada_isolation_forest@champion"
)

predictions = champion_model.predict(
    training_pdf[feature_columns].head(10)
)

print(predictions)

[1 1 1 1 1 1 1 1 1 1]
